# 03 · Entrenamiento RT-DETR Estados Unidos

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Ajuste fino
Ajusta el checkpoint Apache-2.0 con Transformers/PyTorch y exporta automáticamente el resultado a OpenVINO IR; la app ejecuta ese IR con OpenVINO Runtime. Utiliza datos anotados representativos de Estados Unidos y reserva un split independiente para la evaluación final.

In [ ]:
from src.train import train_detector
DATA_DIR=None  # ruta del dataset COCO Estados Unidos
if not DATA_DIR: raise ValueError("Define DATA_DIR con train/ y valid/")
checkpoint=train_detector(DATA_DIR,country="usa",model_id=MODEL_ID,epochs=30,batch_size=8,
                         learning_rate=5e-5,weight_decay=1e-4,seed=42,force=False)
print("Checkpoint:",checkpoint)


## Publicar el IR para el Space (opcional)

El IR queda listo al terminar el entrenamiento. Para que GitHub Actions lo copie a SIRAP_USA_ECUADOR_HF, habilita la celda siguiente después de revisar el modelo. Requiere el secreto `GITHUB_TOKEN` en Colab con permiso de escritura al repositorio.

In [ ]:
from scripts.notebook_io import publish_detector_ir
PUBLISH_IR_TO_SPACE=False  # cambia a True solo cuando quieras publicar este modelo
if PUBLISH_IR_TO_SPACE:
    print(publish_detector_ir(REPO_ROOT, COUNTRY))